# 04 Feature Engineering

This notebook creates explainable customer-level features and exports train/test datasets.

The source code lives in `src.feature_engineering`; this notebook documents the logic and business reasoning.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/Banking and Finance/Projects/canadian-bank-loan-propensity-mlops-deployment')

In [2]:
import pandas as pd

from src.config import CLEANED_DATA_PATH, MODEL_FEATURES, TARGET_COL, TEST_DATA_PATH, TRAIN_DATA_PATH
from src.feature_engineering import (
    engineer_customer_features,
    prepare_modeling_dataset,
    split_features_target,
    create_train_test_split,
    save_train_test_data,
)

cleaned_data = pd.read_csv(CLEANED_DATA_PATH)

## Engineered Features

Features added:

- `MortgageFlag`
- `RelationshipProductCount`
- `DigitalEngagementFlag`
- `SpendPerCustomerYear`
- `MortgageToHighestSpendRatio`
- `ValueSegmentScore`

These features are simple enough to explain to business stakeholders and useful enough for modeling customer propensity.

In [3]:
engineered_data = engineer_customer_features(cleaned_data)

display(engineered_data[MODEL_FEATURES + [TARGET_COL]].head())

,Age,CustomerSince,HighestSpend,MonthlyAverageSpend,Mortgage,Level,HiddenScore,Security,FixedDepositAccount,InternetBanking,CreditCard,MortgageFlag,RelationshipProductCount,DigitalEngagementFlag,SpendPerCustomerYear,MortgageToHighestSpendRatio,ValueSegmentScore,LoanOnCard
0,34,9,180,8.9,0,3,1,0,0,0,0,0,0,0,0.890000,0.0,4,1
1,65,39,105,2.4,0,3,4,0,0,0,0,0,0,0,0.060000,0.0,4,0
2,29,5,45,0.1,0,2,3,0,0,1,0,0,0,1,0.016667,0.0,2,0
3,48,23,114,3.8,0,3,2,1,0,0,0,0,1,0,0.158333,0.0,4,0
4,59,32,40,2.5,0,2,4,0,0,1,0,0,0,1,0.075758,0.0,3,0


In [4]:
modeling_data = prepare_modeling_dataset(cleaned_data)
X, y = split_features_target(modeling_data)

X_train, X_test, y_train, y_test = create_train_test_split(X, y)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Train target rate: {y_train.mean():.2%}")
print(f"Test target rate : {y_test.mean():.2%}")

X_train: (3984, 17)
X_test : (996, 17)
Train target rate: 9.64%
Test target rate : 9.64%


In [5]:
save_train_test_data(X_train, X_test, y_train, y_test)

print(f"Train data saved to: {TRAIN_DATA_PATH}")
print(f"Test data saved to : {TEST_DATA_PATH}")

Train data saved to: D:\Banking and Finance\Projects\canadian-bank-loan-propensity-mlops-deployment\data\processed\03_train_data.csv
Test data saved to : D:\Banking and Finance\Projects\canadian-bank-loan-propensity-mlops-deployment\data\processed\03_test_data.csv


## Feature Engineering Outcome

The model inputs are now reproducible through the source-code pipeline and aligned with API scoring logic.